In [6]:
import numpy as np
import pandas as pd
from utils import precision_recall_by_bin

In [2]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
data = pd.read_csv('../Data/temp_training.csv')

In [13]:
def flag(series, condition):
    return condition.astype(float).where(series.notna())

def divide_positive(numerator, denominator):
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        ratio = numerator / denominator.where(denominator > 0)
    return ratio.replace([np.inf, -np.inf], np.nan)

def divide_all(numerator, denominator):
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        ratio = numerator / denominator
    return ratio.replace([np.inf, -np.inf], np.nan)

In [17]:
# 1. Size
features = data[['id','data_type','default']]
features["log_assets"] = np.log(data["asst_tot"].where(data["asst_tot"] > 0))

# 2. Profitability
features["net_profit_assets"] = divide_positive(data["profit"], data["asst_tot"])
features["net_profit_assets_2"] = divide_all(data["profit"], data["asst_tot"])
features["operating_profit_assets"] = divide_positive(data["prof_operations"], data["asst_tot"])
features["operating_margin"] = divide_positive(data["prof_operations"], data["rev_operating"])

# 3. Liquidity
features["working_capital_assets"] = divide_positive(data["wc_net"], data["asst_tot"])
features["cash_assets"] = divide_positive(data["cash_and_equiv"], data["asst_tot"])
features["cash_short_term_debt"] = divide_positive(data["cash_and_equiv"], data["debt_st"])

# 4. Solvency and debt burden
data['liabilities_tot'] = data["asst_tot"] - data["eqty_tot"]

features["liabilities_assets_proxy"] = divide_positive(data['liabilities_tot'], data["asst_tot"])
features["equity_assets"] = divide_positive(data["eqty_tot"], data["asst_tot"])
features["short_term_debt_assets"] = divide_positive(data["debt_st"], data["asst_tot"])

# 5. Debt-service capacity
features["interest_coverage_proxy"] = divide_positive(data["prof_operations"], data["exp_financing"])

# 6. Cash generation
features["operating_cashflow_assets"] = divide_positive(data["cf_operations"], data["asst_tot"])
features["cashflow_liabilities_proxy"] = divide_positive(data["cf_operations"], data['liabilities_tot'])
features["cashflow_profit_gap_assets"] = divide_positive(data["profit"] - data["cf_operations"], data["asst_tot"])

# 7. Activity and receivables
features["sales_assets"] = divide_positive(data["rev_operating"], data["asst_tot"])
features["receivables_assets"] = divide_positive(data["AR"], data["asst_tot"])
features["receivables_days_proxy"] = 365 * divide_positive(data["AR"].where(data["AR"] >= 0), data["rev_operating"])

# 8. Distress and special-state indicators
features["negative_equity"] = flag(data["eqty_tot"], data["eqty_tot"] < 0)
features["operating_loss"] = flag(data["prof_operations"], data["prof_operations"] < 0)
features["nonpositive_revenue"] = flag(data["rev_operating"], data["rev_operating"] <= 0)
features["zero_financing_expense"] = flag(data["exp_financing"], data["exp_financing"] == 0)

# 9. Business context
features["sector_category"] = data["ateco_sector"].astype("string")
features["legal_structure_category"] = data["legal_struct"].astype("string")

In [18]:
features.isna().sum()

id                                0
data_type                         0
default                           0
log_assets                        0
net_profit_assets                20
net_profit_assets_2              20
operating_profit_assets          21
operating_margin              28012
working_capital_assets            6
cash_assets                      17
cash_short_term_debt           1716
liabilities_assets_proxy          1
equity_assets                     1
short_term_debt_assets            5
interest_coverage_proxy       29651
operating_cashflow_assets        88
cashflow_liabilities_proxy      264
cashflow_profit_gap_assets       88
sales_assets                    171
receivables_assets               12
receivables_days_proxy        28452
negative_equity                   1
operating_loss                   21
nonpositive_revenue             171
zero_financing_expense          362
sector_category                   0
legal_structure_category          0
dtype: int64

In [26]:
features['net_profit_assets'].isna().sum()

np.int64(20)

In [ ]:
bins.groupby()

In [53]:
float(grouped['IV_bin'].sum())

0.7690684100798248

In [ ]:
feature = 'net_profit_assets'


,bins,Goods,Bads,Goods (%),Bads (%),WoE,IV_bin
0,"(-0.00222, 0.000794]",82231,1101,0.100087,9.369500e-02,0.065993,0.000422
1,"(-0.00979, -0.00222]",82308,1025,0.100181,8.722747e-02,0.138454,0.001793
2,"(-0.0309, -0.00979]",82026,1307,0.099837,1.112254e-01,-0.108017,0.001230
3,"(-41.421, -0.0309]",78456,4877,0.095492,4.150294e-01,-1.469306,0.469498
4,"(0.000794, 0.00317]",82337,996,0.100216,8.475959e-02,0.167507,0.002589
5,"(0.00317, 0.00743]",82619,714,0.100559,6.076164e-02,0.503786,0.020049
6,"(0.00743, 0.015]",82753,579,0.100722,4.927325e-02,0.714984,0.036785
7,"(0.015, 0.0292]",82952,381,0.100964,3.242362e-02,1.135880,0.077854
8,"(0.0292, 0.0623]",83025,308,0.101053,2.621139e-02,1.349453,0.100996
9,"(0.0623, 7.936]",82870,463,0.100865,3.940175e-02,0.939968,0.057773


In [35]:
pd.crosstab(bins, features['net_profit_assets'])#.rename(columns={0: 'G', 1: 'B'}) + 0.5

net_profit_assets,-41.420006,-23.329246,-15.885871,-14.601206,-13.190956,-11.401200,-10.906919,-9.159244,-8.972159,-8.230133,...,1.279039,1.381163,1.454709,1.522477,1.656066,2.325271,2.878311,3.587518,5.043156,7.936417
net_profit_assets,,,,,,,,,,,,,,,,,,,,,
"(-0.00222, 0.000794]",0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
"(-0.00979, -0.00222]",0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
"(-0.0309, -0.00979]",0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
"(-41.421, -0.0309]",1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
"(0.000794, 0.00317]",0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
"(0.00317, 0.00743]",0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
"(0.00743, 0.015]",0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
"(0.015, 0.0292]",0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
"(0.0292, 0.0623]",0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
def calculate_bin_iv(df, feature, target, num_bins=5):
    if pd.api.types.is_numeric_dtype(df[feature]):
        bins = pd.qcut(df[feature], q=num_bins, duplicates='drop').astype(str)
        bins = bins.where(df[feature].notna(), "Missing")
    else:
        bins = df[feature].fillna("Missing").astype(str)
        
    # 2. Cross-tabulate and inject a 0.5 laplace smoothing factor to prevent log(0)
    ct = pd.crosstab(bins, df[target]).rename(columns={0: 'G', 1: 'B'}) + 0.5
    
    # 3. Vectorized calculations for Bin-wise WoE and IV
    p_g, p_b = ct['G'] / ct['G'].sum(), ct['B'] / ct['B'].sum()
    ct['WoE'] = np.log(p_g / p_b)
    ct['Bin_IV'] = (p_g - p_b) * ct['WoE']
    
    # Return the total IV scalar and the bin-by-bin analysis breakdown
    return ct['Bin_IV'].sum(), ct[['G', 'B', 'WoE', 'Bin_IV']]


In [ ]:




# 10. Missing financial breakdowns
debt_columns = [
    "debt_bank_st", "debt_bank_lt",
    "debt_fin_st", "debt_fin_lt",
    "AP_st", "AP_lt"
]

features["missing_debt_breakdown"] = (
    data[debt_columns].isna().any(axis=1).astype(int)
)

In [ ]:
def create_features(df):
    f = pd.DataFrame(index=df.index)

    # Divide only when the denominator is positive.
    # Missing inputs and infinite results remain missing.
    

        

    # Missing input should not become a false (0) flag
    



    # Standard current ratio only if debt_st = current liabilities
    f["current_ratio_proxy"] = divide_positive(
        df["asst_current"], df["debt_st"]
    )

    # --------------------------------------------------
    # 4. Solvency and debt burden
    # --------------------------------------------------
    # Accounting identity proxy: verify field definitions first
    liabilities = df["asst_tot"] - df["eqty_tot"]

    f["liabilities_assets_proxy"] = divide_positive(
        liabilities, df["asst_tot"]
    )

    f["equity_assets"] = divide_positive(
        df["eqty_tot"], df["asst_tot"]
    )

    f["short_term_debt_assets"] = divide_positive(
        df["debt_st"], df["asst_tot"]
    )

    # --------------------------------------------------
    # 5. Debt-service capacity
    # --------------------------------------------------
    f["interest_coverage_proxy"] = divide_positive(
        df["prof_operations"], df["exp_financing"]
    )

    # --------------------------------------------------
    # 6. Cash generation
    # --------------------------------------------------
    f["operating_cashflow_assets"] = divide_positive(
        df["cf_operations"], df["asst_tot"]
    )

    f["cashflow_liabilities_proxy"] = divide_positive(
        df["cf_operations"], liabilities
    )

    f["cashflow_profit_gap_assets"] = divide_positive(
        df["profit"] - df["cf_operations"],
        df["asst_tot"]
    )

    # --------------------------------------------------
    # 7. Activity and receivables
    # --------------------------------------------------
    f["sales_assets"] = divide_positive(
        df["rev_operating"], df["asst_tot"]
    )

    f["receivables_assets"] = divide_positive(
        df["AR"], df["asst_tot"]
    )

    f["receivables_days_proxy"] = 365 * divide_positive(
        df["AR"].where(df["AR"] >= 0),
        df["rev_operating"]
    )

    # --------------------------------------------------
    # 8. Distress and special-state indicators
    # --------------------------------------------------
    f["negative_equity"] = flag(
        df["eqty_tot"], df["eqty_tot"] < 0
    )

    f["operating_loss"] = flag(
        df["prof_operations"], df["prof_operations"] < 0
    )

    f["nonpositive_revenue"] = flag(
        df["rev_operating"], df["rev_operating"] <= 0
    )

    f["zero_financing_expense"] = flag(
        df["exp_financing"], df["exp_financing"] == 0
    )

    # --------------------------------------------------
    # 9. Business context
    # --------------------------------------------------
    f["sector_category"] = df["ateco_sector"].astype("string")
    f["legal_structure_category"] = df["legal_struct"].astype("string")

    # --------------------------------------------------
    # 10. Missing financial breakdowns
    # --------------------------------------------------
    debt_columns = [
        "debt_bank_st", "debt_bank_lt",
        "debt_fin_st", "debt_fin_lt",
        "AP_st", "AP_lt"
    ]

    f["missing_debt_breakdown"] = (
        df[debt_columns].isna().any(axis=1).astype(int)
    )

    return f.replace([np.inf, -np.inf], np.nan)

In [2]:
len([-1]*45)

45

In [ ]:
features_list = {
    
}